<a href="https://colab.research.google.com/github/Rads-cw/AI-Support-Ticket-Router/blob/main/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
import pandas as pd

data = {
    "ticket": [
        "I can't log into my account",
        "The app keeps crashing when I open it",
        "I was charged twice for my subscription",
        "I forgot my password and cannot sign in",
        "The website is loading very slowly",
        "My payment was declined",
        "I need to change my email address",
        "The software freezes when I upload a file",
        "I want a refund for my purchase",
        "My account has been locked"
    ],
    "category": [
        "Account",
        "Technical",
        "Billing",
        "Account",
        "Technical",
        "Billing",
        "Account",
        "Technical",
        "Billing",
        "Account"
    ]
}

df = pd.DataFrame(data)
df

,ticket,category
0,I can't log into my account,Account
1,The app keeps crashing when I open it,Technical
2,I was charged twice for my subscription,Billing
3,I forgot my password and cannot sign in,Account
4,The website is loading very slowly,Technical
5,My payment was declined,Billing
6,I need to change my email address,Account
7,The software freezes when I upload a file,Technical
8,I want a refund for my purchase,Billing
9,My account has been locked,Account


In [32]:
!pip install -q datasets

In [33]:
from datasets import load_dataset

dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})


In [34]:
import pandas as pd

data = {
    "ticket": [

        # ACCOUNT
        "I can't log into my account",
        "I forgot my password",
        "My account has been locked",
        "I need to change my email address",
        "I cannot verify my account",
        "My username is not working",
        "I want to update my profile information",
        "I cannot access my account after resetting my password",
        "My account was disabled",
        "I need help recovering my account",
        "The verification code is not arriving",
        "I changed my phone and cannot sign in anymore",
        "How can I change the phone number linked to my profile",
        "I no longer have access to my old email",
        "Someone may have accessed my account",
        "I want to delete my account",
        "My password reset link has expired",
        "I accidentally created two accounts",
        "I cannot complete identity verification",
        "How do I update my account information",
        "My login credentials are being rejected",
        "I was signed out and cannot get back in",
        "My profile information is incorrect",
        "I need to recover my username",
        "The login verification code keeps failing",

        # TECHNICAL
        "The app keeps crashing",
        "The website is loading very slowly",
        "The software freezes when I upload a file",
        "The page is not opening",
        "The application shows an error message",
        "The app closes whenever I click submit",
        "The website is not responding",
        "The program will not install",
        "The application is stuck on the loading screen",
        "I cannot upload documents",
        "The screen turns blank after I open the app",
        "The download button does nothing",
        "The application keeps freezing",
        "Images are not loading on the website",
        "The app stopped working after the latest update",
        "I receive an error whenever I try to save",
        "The website keeps refreshing by itself",
        "My files fail to upload",
        "The software is extremely slow",
        "The app will not open on my computer",
        "Notifications are not appearing",
        "The page crashes when I submit the form",
        "The application cannot connect to the server",
        "The search feature is not working",
        "The website displays a blank page",

        # BILLING
        "I was charged twice",
        "My payment was declined",
        "I want a refund",
        "I was charged the wrong amount",
        "My subscription payment failed",
        "I don't recognize this charge",
        "I was billed after cancelling",
        "My card was charged but the order failed",
        "I need a copy of my invoice",
        "I want to cancel my subscription",
        "Why was I charged after cancelling my plan",
        "I cancelled my subscription but money was still taken",
        "I was billed after I ended my membership",
        "My card was charged even though I cancelled",
        "I cancelled but I still received a charge",
        "Why did you take money from my card",
        "I was charged after cancelling my subscription",
        "I ended my plan but was still billed",
        "The amount on my invoice is incorrect",
        "I have not received my refund",
        "My credit card was charged unexpectedly",
        "Why did my subscription price increase",
        "I was charged for something I did not purchase",
        "My refund has not arrived yet",
        "There is an extra fee on my bill"
    ],

    "category":
        ["Account"] * 25 +
        ["Technical"] * 25 +
        ["Billing"] * 25
}

df = pd.DataFrame(data)

print("Total tickets:", len(df))
print(df["category"].value_counts())

Total tickets: 75
category
Account      25
Technical    25
Billing      25
Name: count, dtype: int64


In [35]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    df["ticket"],
    df["category"],
    test_size=0.25,
    random_state=42,
    stratify=df["category"]
)

# Convert text into TF-IDF features
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2)
)

X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

# Train model
model = LogisticRegression(
    max_iter=1000
)

model.fit(X_train_vectorized, y_train)

print("Training complete!")
print("Training tickets:", len(X_train))
print("Testing tickets:", len(X_test))

Training complete!
Training tickets: 56
Testing tickets: 19


In [36]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Make predictions on unseen test data
y_pred = model.predict(X_test_vectorized)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 94.74 %

Classification Report:
              precision    recall  f1-score   support

     Account       1.00      0.83      0.91         6
     Billing       1.00      1.00      1.00         6
   Technical       0.88      1.00      0.93         7

    accuracy                           0.95        19
   macro avg       0.96      0.94      0.95        19
weighted avg       0.95      0.95      0.95        19


Confusion Matrix:
[[5 0 1]
 [0 6 0]
 [0 0 7]]


In [38]:
X = vectorizer.fit_transform(df["ticket"])
model.fit(X, df["category"])

print("Model retrained!")

Model retrained!


In [56]:
def get_priority(ticket_text, category):
    text = ticket_text.lower()

    account_high = [
        "locked",
        "account locked",
        "cannot access",
        "can't access",
        "unable to access",
        "cannot log in",
        "can't log in",
        "unable to log in",
        "cannot sign in",
        "can't sign in",
        "hacked",
        "compromised",
        "unauthorized access",
        "someone accessed",
        "security breach",
        "verification code not received",
        "verification code not arriving",
        "account disabled",
        "account blocked"
    ]

    technical_high = [
        "crash",
        "crashing",
        "crashed",
        "blank screen",
        "screen goes blank",
        "black screen",
        "freezes",
        "freezing",
        "frozen",
        "not responding",
        "unresponsive",
        "cannot open",
        "can't open",
        "won't open",
        "will not open",
        "cannot load",
        "won't load",
        "stopped working",
        "not working",
        "server down",
        "offline",
        "cannot save",
        "save failed",
        "cannot upload",
        "upload failed",
        "lost data",
        "data lost",
        "deleted files",
        "missing files"
    ]

    billing_high = [
        "charged twice",
        "double charged",
        "duplicate charge",
        "charged again",
        "still charged",
        "charged after",
        "billed after",
        "billed again",
        "unauthorized charge",
        "unrecognized charge",
        "charged unexpectedly",
        "incorrect charge",
        "wrong amount",
        "money taken",
        "money deducted",
        "payment failed",
        "payment declined",
        "refund not received",
        "refund missing",
        "refund has not arrived"
    ]

    general_high = [
        "urgent",
        "immediately",
        "as soon as possible",
        "asap",
        "emergency",
        "critical",
        "need help now"
    ]

    if any(word in text for word in general_high):
        return "High"

    if category == "Account" and any(word in text for word in account_high):
        return "High"

    if category == "Technical" and any(word in text for word in technical_high):
        return "High"

    if category == "Billing" and any(word in text for word in billing_high):
        return "High"

    return "Normal"


ticket_text = input("Enter a support ticket: ")

ticket_vector = vectorizer.transform([ticket_text])
category = model.predict(ticket_vector)[0]

department_map = {
    "Account": "Account Support",
    "Technical": "Technical Support",
    "Billing": "Billing Support"
}

department = department_map[category]

priority = get_priority(ticket_text, category)

print("\n--- Ticket Analysis ---")
print("Category:", category)
print("Priority:", priority)
print("Assigned Team:", department)

Enter a support ticket: I cancelled my subscription last week, but I was charged again today.

--- Ticket Analysis ---
Category: Billing
Priority: High
Assigned Team: Billing Support


In [57]:
!pip install -q gradio

In [62]:
import gradio as gr

department_map = {
    "Account": "Account Support",
    "Technical": "Technical Support",
    "Billing": "Billing Support"
}

def analyze_ticket(ticket_text):
    if not ticket_text.strip():
        return "Please enter a support ticket.", "", ""

    ticket_vector = vectorizer.transform([ticket_text])
    category = model.predict(ticket_vector)[0]

    department = department_map.get(category, "General Support")
    priority = get_priority(ticket_text, category)

    return category, priority, department


demo = gr.Interface(
    fn=analyze_ticket,

    inputs=gr.Textbox(
        lines=6,
        label="Support Ticket",
        placeholder="Describe the customer's issue..."
    ),

    outputs=[
        gr.Textbox(label="Category"),
        gr.Textbox(label="Priority"),
        gr.Textbox(label="Assigned Team")
    ],

    title="AI Support Ticket Router",

    description=(
        "Enter a customer support ticket to automatically classify, "
        "prioritize, and route it."
    ),

    # Keep the Flag button visible
    flagging_mode="manual",

    # Let the user explain WHY they flagged it
    flagging_options=[
        "Wrong Category",
        "Wrong Priority",
        "Wrong Team",
        "Other"
    ],

    # Save flagged tickets here
    flagging_dir="flagged_tickets",

    # Save everything into one CSV
    flagging_callback=gr.CSVLogger(
        dataset_file_name="flagged_tickets.csv"
    )
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://198e43254cdd386085.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
